# Step 0: Oracle labels for the 460-document subset
Turns each abstractive reference summary into per-sentence extractive labels (binary `y` and soft) used to train the GNN scorer.

**Before running:** edit the paths in cell 2. Everything else is automatic. The splits are your existing `train / validate / test` folders, so no re-splitting happens here.

In [1]:
!pip -q install rouge-score tqdm
from google.colab import drive
drive.mount('/content/drive')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


In [3]:
import os, re, json, time
from pathlib import Path
from collections import Counter
from multiprocessing import Pool, cpu_count
from tqdm.auto import tqdm

# ============ EDIT THESE ============
BASE         = "/content/drive/MyDrive/inabs_project"      # your project folder
NER_DIR      = f"{BASE}/ner_v2"                           # contains train/ validate/ test/ with the NER .json files
EMB_DIR      = f"{BASE}/embeddings"                    # contains train/ validate/ test/ with .pt files (None = skip check)
SUMMARY_ROOT = "/content/drive/MyDrive/inabs_project/summaries"         # folder holding the reference summaries (searched recursively)
SUMMARY_PATH_KEYWORD = None                       # only use .txt files whose path contains this word
                                                       # (IN-Abs has judgement/ and summary/ folders with the same file names)
LABEL_DIR    = f"{BASE}/labels"                        # output
SPLIT_NAMES  = ["train", "validate", "test"]           # subfolder names
# ====================================

MAX_ORACLE_SENTS = 100    # cap on oracle sentences per document
PREFILTER_K      = None   # e.g. 80 to restrict greedy to the top-80 sentences by ROUGE-2 recall; None = use all (fast enough)
OVERWRITE        = False  # False = skip documents already labelled (safe to resume after a disconnect)

## 1. Collect the 460 processed documents and match their reference summaries

In [4]:
def norm_key(name):
    """file name -> comparable key (drop one extension, lowercase)."""
    return re.sub(r"\.[A-Za-z0-9]{1,5}$", "", name.strip()).lower()

# --- index reference summaries
summary_index, dupes = {}, 0
for p in Path(SUMMARY_ROOT).rglob("*.txt"):
    rel = str(p.relative_to(SUMMARY_ROOT)).lower()
    if SUMMARY_PATH_KEYWORD and SUMMARY_PATH_KEYWORD.lower() not in rel:
        continue
    k = norm_key(p.name)
    if k in summary_index: dupes += 1
    summary_index[k] = p
print(f"Indexed {len(summary_index)} summary files ({dupes} duplicate names)")

# --- load the processed docs (ONLY those present in your NER folders)
docs, issues = [], {"no_summary": [], "emb_mismatch": [], "bad_json": []}
for split in SPLIT_NAMES:
    for f in sorted(Path(NER_DIR, split).glob("*.json")):
        try:
            d = json.load(open(f, encoding="utf-8"))
            sents = [s["text"] for s in sorted(d["sentences"], key=lambda s: s["sent_id"])]
        except Exception as e:
            issues["bad_json"].append((str(f), repr(e))); continue
        fname = d.get("filename") or f.stem
        sp = summary_index.get(norm_key(fname)) or summary_index.get(norm_key(f.stem))
        if sp is None:
            issues["no_summary"].append((split, f.stem, fname)); continue
        ref = re.sub(r"\s+", " ", sp.read_text(encoding="utf-8", errors="ignore")).strip()

        if EMB_DIR:   # alignment check: one embedding row per sentence
            import torch
            ep = Path(EMB_DIR, split, f.stem + ".pt")
            if ep.exists():
                obj = torch.load(ep, map_location="cpu", weights_only=False)
                emb = obj["embeddings"] if isinstance(obj, dict) else obj
                if emb.shape[0] != len(sents):
                    issues["emb_mismatch"].append((split, f.stem, emb.shape[0], len(sents))); continue

        docs.append(dict(split=split, doc_id=f.stem, filename=fname, sents=sents, ref=ref))

print({s: sum(d["split"] == s for d in docs) for s in SPLIT_NAMES}, "usable docs")
for k, v in issues.items():
    print(f"{k}: {len(v)}", v[:3])
if issues["no_summary"]:
    print("\nSample summary keys :", list(summary_index)[:3])
    print("Unmatched filenames :", [x[2] for x in issues["no_summary"][:3]])

Indexed 7128 summary files (0 duplicate names)
{'train': 300, 'validate': 60, 'test': 100} usable docs
no_summary: 0 []
emb_mismatch: 0 []
bad_json: 0 []


## 2. Oracle labeler (greedy, ROUGE-1/2 with the same tokenizer + stemming as `rouge-score`)
N-gram counts are cached per sentence and gains are computed incrementally, so greedy over *all* sentences is fast and no ROUGE-2 pre-filter is needed.

In [5]:
from rouge_score import rouge_scorer, tokenizers
TOK    = tokenizers.DefaultTokenizer(use_stemmer=True)
SCORER = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def ngram_counts(toks, n):
    return Counter(zip(*[toks[i:] for i in range(n)])) if len(toks) >= n else Counter()

def f1(ov, cand_total, ref_total):
    if ov <= 0 or cand_total <= 0 or ref_total <= 0: return 0.0
    p, r = ov / cand_total, ov / ref_total
    return 2 * p * r / (p + r)

def gain(sel, cand, ref):
    """Increase in clipped overlap if `cand` is added to `sel`."""
    return sum(min(sel.get(g, 0) + c, ref[g]) - min(sel.get(g, 0), ref[g]) for g, c in cand.items())

def process_doc(job):
    sents, ref_text = job["sents"], job["ref"]
    n = len(sents)
    ref_t = TOK.tokenize(ref_text)
    ref1, ref2 = ngram_counts(ref_t, 1), ngram_counts(ref_t, 2)
    R1, R2 = len(ref_t), max(len(ref_t) - 1, 0)

    # per-sentence caches (counts restricted to n-grams that occur in the reference)
    L1, L2, c1, c2, soft = [], [], [], [], []
    for s in sents:
        t = TOK.tokenize(s)
        a, b = ngram_counts(t, 1), ngram_counts(t, 2)
        L1.append(len(t)); L2.append(max(len(t) - 1, 0))
        c1.append(Counter({g: c for g, c in a.items() if g in ref1}))
        c2.append(Counter({g: c for g, c in b.items() if g in ref2}))
        ov2 = sum(min(c, ref2[g]) for g, c in c2[-1].items())
        soft.append(ov2 / R2 if R2 else 0.0)          # per-sentence ROUGE-2 recall

    cand = list(range(n))
    if PREFILTER_K and n > PREFILTER_K:
        cand = sorted(cand, key=lambda i: -soft[i])[:PREFILTER_K]

    # greedy selection on (R1-F + R2-F) / 2
    sel1, sel2, ov1, ov2, t1, t2 = Counter(), Counter(), 0, 0, 0, 0
    chosen, best_score, remaining = [], 0.0, set(cand)
    while remaining and len(chosen) < MAX_ORACLE_SENTS:
        best = None
        for i in remaining:
            if L1[i] == 0: continue
            g1, g2 = gain(sel1, c1[i], ref1), gain(sel2, c2[i], ref2)
            sc = (f1(ov1 + g1, t1 + L1[i], R1) + f1(ov2 + g2, t2 + L2[i], R2)) / 2
            if sc > best_score + 1e-9 and (best is None or sc > best[0]):
                best = (sc, i, g1, g2)
        if best is None: break                          # nothing improves -> stop
        best_score, i, g1, g2 = best
        chosen.append(i); remaining.discard(i)
        sel1.update(c1[i]); sel2.update(c2[i])
        ov1 += g1; ov2 += g2; t1 += L1[i]; t2 += L2[i]

    chosen_sorted = sorted(chosen)
    y = [1 if i in set(chosen) else 0 for i in range(n)]
    mx, mn = max(soft), min(soft)
    soft_n = [(v - mn) / (mx - mn) if mx > mn else 0.0 for v in soft]

    # official rouge-score on the final oracle summary (reporting only)
    sc = SCORER.score(ref_text, " ".join(sents[i] for i in chosen_sorted)) if chosen else None
    return dict(
        doc_id=job["doc_id"], filename=job["filename"], split=job["split"],
        num_sentences=n, y=y, soft=[round(v, 5) for v in soft_n],
        selected_idx=chosen_sorted, n_selected=len(chosen),
        ref_words=len(ref_text.split()),
        oracle_rouge={k: round(v.fmeasure, 4) for k, v in sc.items()} if sc else {"rouge1": 0, "rouge2": 0, "rougeL": 0},
        reference=ref_text,
    )

## 3. Run on all 460 documents (parallel, resumable)

In [6]:
def out_path(d): return Path(LABEL_DIR, d["split"], d["doc_id"] + ".json")

for s in SPLIT_NAMES: Path(LABEL_DIR, s).mkdir(parents=True, exist_ok=True)
todo = [d for d in docs if OVERWRITE or not out_path(d).exists()]
print(f"{len(docs)} docs, {len(todo)} to label, {cpu_count()} CPUs")

t0 = time.time()
if todo:
    with Pool(cpu_count()) as pool:
        for res in tqdm(pool.imap_unordered(process_doc, todo, chunksize=1), total=len(todo)):
            json.dump(res, open(out_path(res), "w", encoding="utf-8"))
print(f"done in {time.time() - t0:.0f}s")

# manifest of exactly which documents belong to which split (reuse this in every later step)
manifest = {s: sorted(d["doc_id"] for d in docs if d["split"] == s) for s in SPLIT_NAMES}
json.dump(manifest, open(Path(LABEL_DIR, "splits.json"), "w"), indent=1)

460 docs, 460 to label, 12 CPUs


  0%|          | 0/460 [00:00<?, ?it/s]

done in 85s


## 4. Stats and sanity checks
The mean reference length (words) is what you will use for the MMR word budget (Step 4) and LED `max_length` (Step 5). Oracle ROUGE is your upper bound.

In [7]:
import numpy as np
rows = [json.load(open(p)) for s in SPLIT_NAMES for p in sorted(Path(LABEL_DIR, s).glob("*.json"))]
rows = [r for r in rows if r["doc_id"] in {d["doc_id"] for d in docs}]   # restrict to the 460

def fround(x, k):
    return round(float(x), k)

def summarize(rs):
    return dict(
        docs=len(rs),
        mean_sents=fround(np.mean([r["num_sentences"] for r in rs]), 1),
        mean_ref_words=fround(np.mean([r["ref_words"] for r in rs]), 1),
        mean_oracle_sents=fround(np.mean([r["n_selected"] for r in rs]), 1),
        pos_rate=fround(np.mean([sum(r["y"]) / r["num_sentences"] for r in rs]), 3),
        oracle_R1=fround(np.mean([r["oracle_rouge"]["rouge1"] for r in rs]), 4),
        oracle_R2=fround(np.mean([r["oracle_rouge"]["rouge2"] for r in rs]), 4),
        oracle_RL=fround(np.mean([r["oracle_rouge"]["rougeL"] for r in rs]), 4),
    )

stats = {s: summarize([r for r in rows if r["split"] == s]) for s in SPLIT_NAMES}
stats["all"] = summarize(rows)
stats["ref_words_p10_p50_p90"] = [int(np.percentile([r["ref_words"] for r in rows], q)) for q in (10, 50, 90)]
json.dump(stats, open(Path(LABEL_DIR, "stats.json"), "w"), indent=1)
for k, v in stats.items(): print(k, v)

# red flags
zero  = [r["doc_id"] for r in rows if r["n_selected"] == 0]
capped = [r["doc_id"] for r in rows if r["n_selected"] >= MAX_ORACLE_SENTS]
lowR2 = [r["doc_id"] for r in rows if r["oracle_rouge"]["rouge2"] < 0.10]
print(f"\nno positives: {len(zero)} | hit the sentence cap: {len(capped)} | oracle R2 < 0.10: {len(lowR2)}")

# eyeball one document
r = rows[0]
print("\nREFERENCE:", r["reference"][:400], "...")
print("\nORACLE SENTENCES:")
for i in r["selected_idx"][:5]:
    print(f"  [{i}]", json.load(open(next(Path(NER_DIR, r['split']).glob(r['doc_id'] + '.json'))))["sentences"][i]["text"][:200])

train {'docs': 300, 'mean_sents': 94.6, 'mean_ref_words': 898.1, 'mean_oracle_sents': 17.0, 'pos_rate': 0.212, 'oracle_R1': 0.6999, 'oracle_R2': 0.5234, 'oracle_RL': 0.4619}
validate {'docs': 60, 'mean_sents': 83.7, 'mean_ref_words': 742.0, 'mean_oracle_sents': 15.2, 'pos_rate': 0.212, 'oracle_R1': 0.6981, 'oracle_R2': 0.5145, 'oracle_RL': 0.45}
test {'docs': 100, 'mean_sents': 87.0, 'mean_ref_words': 800.9, 'mean_oracle_sents': 16.1, 'pos_rate': 0.216, 'oracle_R1': 0.7103, 'oracle_R2': 0.533, 'oracle_RL': 0.4659}
all {'docs': 460, 'mean_sents': 91.5, 'mean_ref_words': 856.6, 'mean_oracle_sents': 16.6, 'pos_rate': 0.213, 'oracle_R1': 0.7019, 'oracle_R2': 0.5243, 'oracle_RL': 0.4612}
ref_words_p10_p50_p90 [294, 619, 1506]

no positives: 0 | hit the sentence cap: 2 | oracle R2 < 0.10: 0

REFERENCE: TRAIN - Summary 1007 One Sharda Singh, respondent, who was an employee of the appellant mills was dismissed for disobeying the orders of the managing authority. He filed an application before 